In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import sys
BASE_DIR = '/content/drive/MyDrive/chess_engine/data'
RAW_DIR = '/content/drive/MyDrive/chess_engine/data/raw/lichess_db_eval.jsonl.zst'
PROC_DIR = os.path.join(BASE_DIR,'process')
os.makedirs(PROC_DIR,exist_ok=True)
RAW_DIR, PROC_DIR

In [ ]:
!pip -q install zstandard orjson tqdm


In [ ]:
import os, time, math, zlib
import numpy as np
import zstandard as zstd
import orjson
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, '/content/drive/MyDrive/chess_engine/')

from core.board import Board
from representation.encode import encode_v2

# ================== CONFIG ==================
CP_SCALE = 600.0
FIXED_DEPTH = 25
MIN_KNODES = 50_000
DEPTH_POLICY = "at_least"   # "exact" hoặc "at_least"

# POV config:
# - SOURCE_CP_POV: POV của cp/mate trong file JSONL ("white" hoặc "stm")
# - LABEL_POV: POV của label dùng để train
SOURCE_CP_POV = "white"   # lichess_db_eval thường là white POV
LABEL_POV = "stm"         # phù hợp nếu encode_board là STM-relative

# Legacy shuffle buffer (kept for backward compatibility).
# Với balanced direct shard writer bên dưới, buffer này không còn là thành phần chính.
SHUFFLE_BUFFER_SIZE = 20_000

# Opening filter (FEN-level)
MAX_PIECES_OPENING = 28
PHASE_OPENING_MIN = 20
PHASE_W = {'N': 1, 'B': 1, 'R': 2, 'Q': 4}

# Mate / extreme filter
MIN_MATE_DISTANCE = 3
MAX_ABS_CP = 1200
KEEP_MATE_PROB = 0.10
SAMPLE_PROB = 1.0

# Dirty-sample filter: drop FENs where |y| is near zero but material is heavily imbalanced
# This catches repetition draws, fortresses, and stalemate traps
DIRTY_Y_THRESHOLD = 0.05       # |y| < this -> candidate for filtering
DIRTY_MAT_THRESHOLD = 2.0      # material_delta raw (in pawn units, before /39 normalization)

# Target output
TARGET_TOTAL = 5_000_000
SHARD_SIZE = 50_000
SPLIT_RATIO = {"train": 0.8, "val": 0.1, "test": 0.1}

# Distribution bins for y in [-1, 1]
N_BUCKETS = 20
BUCKET_EDGES = np.linspace(-1.0, 1.0, N_BUCKETS + 1)

# Chess-like target distribution (đối xứng quanh 0)
TARGET_BUCKET_WEIGHTS = np.array([
    0.014, 0.022, 0.034, 0.035, 0.029,
    0.027, 0.029, 0.035, 0.060, 0.215,
    0.215, 0.060, 0.035, 0.029, 0.027,
    0.029, 0.035, 0.034, 0.022, 0.014
], dtype=np.float64)
TARGET_BUCKET_WEIGHTS = TARGET_BUCKET_WEIGHTS / TARGET_BUCKET_WEIGHTS.sum()

# Apply bucket quota per split để tránh Train/Val distribution mismatch
APPLY_BUCKET_QUOTA = {
    "train": True,
    "val": True,
    "test": True,
}


def ensure_full_fen(fen: str) -> str:
    parts = fen.split()
    return (fen + " 0 1") if len(parts) == 4 else fen


def canonical_fen_for_split(fen: str) -> str:
    """
    Canonical key cho split để tránh data leakage từ halfmove/fullmove.
    Luôn giữ 4 trường đầu: board, side-to-move, castling, en-passant.
    """
    parts = fen.split()
    if len(parts) >= 4:
        return " ".join(parts[:4])
    return fen.strip()


def split_by_fen(fen: str, train=0.8, val=0.1) -> str:
    key = canonical_fen_for_split(fen)
    h = zlib.crc32(key.encode("utf-8")) & 0xffffffff
    r = h / 2**32
    if r < train:
        return "train"
    elif r < train + val:
        return "val"
    return "test"


def pick_eval_by_depth(evals, fixed_depth: int, min_knodes: int, policy: str = DEPTH_POLICY):
    cands = []
    for e in evals:
        depth = int(e.get("depth", -1))
        if policy == "exact":
            if depth != fixed_depth:
                continue
        elif policy == "at_least":
            if depth < fixed_depth:
                continue
        else:
            raise ValueError(f"Unsupported DEPTH_POLICY: {policy}")

        if int(e.get("knodes", 0)) < min_knodes:
            continue

        pvs = e.get("pvs")
        if not pvs:
            continue

        cands.append(e)

    if not cands:
        return None

    # Ưu tiên depth cao hơn trước, rồi mới đến knodes
    return max(cands, key=lambda x: (int(x.get("depth", -1)), int(x.get("knodes", -1))))


def mate_to_cp(mate: int) -> float:
    sign = 1.0 if mate > 0 else -1.0
    m = min(abs(int(mate)), 100)
    return sign * (10000.0 - 100.0 * m)


def pv_passes_mate_and_extreme_filters(pv) -> bool:
    if "mate" in pv:
        return abs(int(pv["mate"])) >= MIN_MATE_DISTANCE
    if "cp" in pv:
        return abs(int(pv["cp"])) <= MAX_ABS_CP
    return False


def fen_board_piece_count_and_phase(board_part: str):
    pieces = 0
    phase = 0
    for ch in board_part:
        if ch == '/' or ch.isdigit():
            continue
        pieces += 1
        phase += PHASE_W.get(ch.upper(), 0)
    return pieces, phase


def is_opening_fen(fen: str) -> bool:
    board_part = fen.split()[0]
    pieces, phase = fen_board_piece_count_and_phase(board_part)
    return (pieces > MAX_PIECES_OPENING) and (phase >= PHASE_OPENING_MIN)


def cp_from_source_to_white(cp: float, stm: str, source_pov: str) -> float:
    if source_pov == "white":
        return cp
    if source_pov == "stm":
        return cp if stm == "w" else -cp
    raise ValueError(f"Unsupported SOURCE_CP_POV: {source_pov}")


def cp_from_white_to_label(cp_white: float, stm: str, label_pov: str) -> float:
    if label_pov == "white":
        return cp_white
    if label_pov == "stm":
        return cp_white if stm == "w" else -cp_white
    raise ValueError(f"Unsupported LABEL_POV: {label_pov}")


def pv_to_label_value(pv, stm: str, source_cp_pov: str = SOURCE_CP_POV, label_pov: str = LABEL_POV) -> float:
    if "cp" in pv:
        cp_src = float(pv["cp"])
    else:
        cp_src = mate_to_cp(int(pv["mate"]))

    cp_white = cp_from_source_to_white(cp_src, stm=stm, source_pov=source_cp_pov)
    cp_label = cp_from_white_to_label(cp_white, stm=stm, label_pov=label_pov)
    return math.tanh(cp_label / CP_SCALE)


def to_channels_first(X: np.ndarray) -> np.ndarray:
    if X.shape in ((18, 8, 8), (23, 8, 8)):
        return X
    if X.shape in ((8, 8, 18), (8, 8, 23)):
        return np.transpose(X, (2, 0, 1))
    raise ValueError(f"Unexpected encode shape: {X.shape}")


def detect_encode_mode() -> str:
    """
    Kiểm tra encode_board đang theo POV nào dựa trên own/opponent planes (0-5 vs 6-11).
    Trả về: "stm", "white", hoặc "unknown".
    """
    fen_w = "8/8/8/8/8/8/PP6/k6K w - - 0 1"
    fen_b = "8/8/8/8/8/8/PP6/k6K b - - 0 1"

    Xw = to_channels_first(encode_v2(Board(fen_w)))
    Xb = to_channels_first(encode_v2(Board(fen_b)))

    own_w, opp_w = float(Xw[:6].sum()), float(Xw[6:12].sum())
    own_b, opp_b = float(Xb[:6].sum()), float(Xb[6:12].sum())

    # Với fen test này: white pieces > black pieces
    # - Nếu stm-relative: own_w > opp_w và own_b < opp_b
    # - Nếu white-relative: own_w > opp_w và own_b > opp_b
    if own_w > opp_w and own_b < opp_b:
        return "stm"
    if own_w > opp_w and own_b > opp_b:
        return "white"
    return "unknown"


ENCODE_MODE = detect_encode_mode()
if ENCODE_MODE == "unknown":
    print("WARNING: Could not infer encode mode automatically.")
elif ENCODE_MODE != LABEL_POV:
    print(f"WARNING: LABEL_POV={LABEL_POV} but encode_board looks like {ENCODE_MODE} POV.")


def bucket_id(y: float, edges: np.ndarray) -> int:
    i = int(np.searchsorted(edges, y, side="right") - 1)
    if i < 0:
        return 0
    if i >= len(edges) - 1:
        return len(edges) - 2
    return i


def build_bucket_quota(total: int, weights: np.ndarray) -> np.ndarray:
    w = np.asarray(weights, dtype=np.float64)
    w = w / w.sum()
    raw = total * w
    q = np.floor(raw).astype(np.int64)
    rem = int(total - q.sum())
    if rem > 0:
        frac = raw - q
        idx = np.argsort(-frac)[:rem]
        q[idx] += 1
    return q


def shard_caps_from_total(total: int, shard_size: int) -> np.ndarray:
    if total <= 0:
        return np.zeros((0,), dtype=np.int64)
    n = int((total + shard_size - 1) // shard_size)
    caps = np.full((n,), int(shard_size), dtype=np.int64)
    caps[-1] = int(total - shard_size * (n - 1))
    return caps


def allocate_bucket_quota_to_shards(bucket_quota: np.ndarray, shard_caps: np.ndarray) -> np.ndarray:
    """
    Allocate exact per-bucket quota to each shard.
    Returns matrix Q[shard, bucket] with:
      - Q.sum(axis=0) == bucket_quota
      - Q.sum(axis=1) == shard_caps
    """
    bucket_quota = np.asarray(bucket_quota, dtype=np.int64)
    shard_caps = np.asarray(shard_caps, dtype=np.int64)

    if int(bucket_quota.sum()) != int(shard_caps.sum()):
        raise ValueError(
            f"Bucket quota sum ({int(bucket_quota.sum())}) != shard caps sum ({int(shard_caps.sum())})"
        )

    n_shards = int(shard_caps.shape[0])
    n_buckets = int(bucket_quota.shape[0])

    out = np.zeros((n_shards, n_buckets), dtype=np.int64)
    remain_col = bucket_quota.copy()

    for i in range(n_shards):
        row_need = int(shard_caps[i])
        if row_need <= 0:
            continue

        remain_total = int(remain_col.sum())
        if remain_total < row_need:
            raise RuntimeError("Not enough remaining bucket quota to fill shard row")

        if i == n_shards - 1:
            out[i] = remain_col
            remain_col[:] = 0
            break

        if remain_total == 0:
            continue

        w = remain_col.astype(np.float64)
        w = w / w.sum()
        row = build_bucket_quota(row_need, w).astype(np.int64)

        # clip row by remaining per bucket
        row = np.minimum(row, remain_col)
        allocated = int(row.sum())

        if allocated < row_need:
            need = row_need - allocated
            room = remain_col - row
            order = np.argsort(-room)
            for b in order:
                if need <= 0:
                    break
                add = min(int(room[b]), need)
                row[b] += add
                need -= add

            if need != 0:
                raise RuntimeError("Cannot distribute leftover row quota")

        out[i] = row
        remain_col -= row

    if int(remain_col.sum()) != 0:
        raise RuntimeError("Column quota not fully allocated across shards")

    if not np.all(out.sum(axis=1) == shard_caps):
        raise RuntimeError("Row sums mismatch in shard quota allocation")

    if not np.all(out.sum(axis=0) == bucket_quota):
        raise RuntimeError("Column sums mismatch in shard quota allocation")

    return out


def save_hist_png(y_values, edges, out_path, title):
    plt.figure()
    plt.hist(y_values, bins=edges)
    plt.title(title)
    plt.xlabel("y")
    plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def compute_raw_material_delta(fen: str) -> float:
    """
    Compute raw material delta (STM perspective, in pawn units) from FEN string.
    Returns (material_self - material_opp) where self = side-to-move.
    Pawn=1, Knight=3, Bishop=3, Rook=5, Queen=9, King=0.
    """
    piece_values = {
        'P': 1, 'N': 3, 'B': 3, 'R': 5, 'Q': 9, 'K': 0,
        'p': 1, 'n': 3, 'b': 3, 'r': 5, 'q': 9, 'k': 0
    }
    parts = fen.split()
    board_part = parts[0]
    stm = parts[1] if len(parts) > 1 else "w"

    white_mat = 0.0
    black_mat = 0.0
    for ch in board_part:
        if ch in piece_values:
            if ch.isupper():
                white_mat += piece_values[ch]
            else:
                black_mat += piece_values[ch]

    if stm == "w":
        return white_mat - black_mat
    return black_mat - white_mat


def is_dirty_sample(y: float, fen: str) -> bool:
    """
    Returns True if the sample should be DROPPED.
    Criteria: |y| is near zero (draw-ish eval) but material is heavily imbalanced.
    This catches repetition draws, fortresses, stalemate traps, etc.
    """
    if abs(y) >= DIRTY_Y_THRESHOLD:
        return False
    raw_delta = compute_raw_material_delta(fen)
    return abs(raw_delta) > DIRTY_MAT_THRESHOLD




In [ ]:
# Split quota tổng
quota_split = {
    "train": int(TARGET_TOTAL * SPLIT_RATIO["train"]),
    "val":   int(TARGET_TOTAL * SPLIT_RATIO["val"]),
}
quota_split["test"] = TARGET_TOTAL - quota_split["train"] - quota_split["val"]

# Bucket quota per split
split_bucket_quota = {}
for sp, n in quota_split.items():
    if APPLY_BUCKET_QUOTA.get(sp, False):
        split_bucket_quota[sp] = build_bucket_quota(n, TARGET_BUCKET_WEIGHTS)
    else:
        split_bucket_quota[sp] = None

print("quota_split:", quota_split)
print("FIXED_DEPTH:", FIXED_DEPTH, "DEPTH_POLICY:", DEPTH_POLICY, "MIN_KNODES:", MIN_KNODES)
print("SOURCE_CP_POV:", SOURCE_CP_POV, "LABEL_POV:", LABEL_POV, "ENCODE_MODE:", ENCODE_MODE)
print("SHUFFLE_BUFFER_SIZE (legacy, ignored in reservoir mode):", SHUFFLE_BUFFER_SIZE)
print("APPLY_BUCKET_QUOTA:", APPLY_BUCKET_QUOTA)
for sp in ["train", "val", "test"]:
    q = split_bucket_quota[sp]
    if q is None:
        print(f"{sp} bucket quota: None (natural)")
    else:
        print(f"{sp} bucket quota sum:", int(q.sum()))
        print(f"{sp} bucket quota:", q.tolist())




In [ ]:
def preprocess_mixed(
    zst_path: str,
    out_dir: str,
    shard_size: int,
    quota_split: dict,
    bucket_edges: np.ndarray,
    split_bucket_quota: dict,
    fixed_depth: int = FIXED_DEPTH,
    min_knodes: int = MIN_KNODES,
    depth_policy: str = DEPTH_POLICY,
    shuffle_buffer_size: int = SHUFFLE_BUFFER_SIZE,
    seed: int = 123,
 ):
    """
    Two-pass preprocess:
    - Pass 1: stream JSONL and do per-split per-bucket reservoir sampling (order-unbiased).
    - Pass 2: global shuffle retained pools, then write final shards.

    This removes stream-order bias and avoids pathological last shards.
    """
    from numpy.lib.format import open_memmap
    import glob
    import json as _json
    import shutil

    if ENCODE_MODE != "unknown" and LABEL_POV != ENCODE_MODE:
        raise ValueError(
            f"LABEL_POV={LABEL_POV} xung dot encode mode={ENCODE_MODE}. "
            "Hay dong bo LABEL_POV voi representation.encode.encode_board."
        )

    if int(shuffle_buffer_size) > 1:
        print(
            f"WARNING: SHUFFLE_BUFFER_SIZE={shuffle_buffer_size} is ignored in "
            "two-pass reservoir mode."
        )

    os.makedirs(out_dir, exist_ok=True)

    pool_dir = os.path.join(out_dir, "_tmp_pool")
    stage_dir = os.path.join(out_dir, "_stage_output")
    status_path = os.path.join(out_dir, "_build_status.json")

    def _write_status(status: str, extra=None):
        payload = {
            "status": status,
            "time": time.time(),
            "writer_mode": "two_pass_bucket_reservoir_global_shuffle",
            "zst_path": zst_path,
        }
        if extra:
            payload.update(extra)
        tmp = status_path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            _json.dump(payload, f)
        os.replace(tmp, status_path)

    # clean previous temp dirs
    for p in [pool_dir, stage_dir]:
        if os.path.exists(p):
            shutil.rmtree(p)

    # clean final split dirs (prevent stale shards)
    for sp in quota_split:
        split_dir = os.path.join(out_dir, sp)
        if os.path.exists(split_dir):
            shutil.rmtree(split_dir)

    os.makedirs(pool_dir, exist_ok=True)
    os.makedirs(stage_dir, exist_ok=True)

    rng = np.random.default_rng(seed)
    n_buckets = len(bucket_edges) - 1

    split_state = {}

    def _open_split_pool(sp: str):
        total = int(quota_split[sp])
        if total < 0:
            raise ValueError(f"Invalid quota for split {sp}: {total}")

        sp_pool_dir = os.path.join(pool_dir, sp)
        os.makedirs(sp_pool_dir, exist_ok=True)

        X_path = os.path.join(sp_pool_dir, f"pool_X_{sp}.npy")
        y_path = os.path.join(sp_pool_dir, f"pool_y_{sp}.npy")

        X_pool = open_memmap(X_path, mode="w+", dtype=np.float16, shape=(total, 23, 8, 8))
        y_pool = open_memmap(y_path, mode="w+", dtype=np.float32, shape=(total,))

        q = split_bucket_quota.get(sp)
        if q is None:
            # Global reservoir for split if no bucket quota requested.
            return {
                "mode": "global",
                "total": total,
                "X_pool": X_pool,
                "y_pool": y_pool,
                "seen_total": 0,
                "fill_total": 0,
            }

        q = np.asarray(q, dtype=np.int64)
        if q.shape[0] != n_buckets:
            raise ValueError(f"Split {sp}: bucket quota len {q.shape[0]} != n_buckets {n_buckets}")
        if int(q.sum()) != total:
            raise ValueError(f"Split {sp}: bucket quota sum {int(q.sum())} != split quota {total}")

        start = np.zeros(n_buckets, dtype=np.int64)
        if n_buckets > 1:
            start[1:] = np.cumsum(q[:-1])

        return {
            "mode": "bucket",
            "total": total,
            "X_pool": X_pool,
            "y_pool": y_pool,
            "bucket_cap": q,
            "bucket_start": start,
            "seen_bucket": np.zeros(n_buckets, dtype=np.int64),
            "fill_bucket": np.zeros(n_buckets, dtype=np.int64),
        }

    for sp in quota_split:
        split_state[sp] = _open_split_pool(sp)

    def _close_pool_maps():
        for st in split_state.values():
            for key in ["X_pool", "y_pool"]:
                mm = st.get(key)
                if mm is None:
                    continue
                try:
                    mm.flush()
                except Exception:
                    pass
                try:
                    if hasattr(mm, "_mmap") and (mm._mmap is not None):
                        mm._mmap.close()
                except Exception:
                    pass
                st[key] = None

    stats = {
        "seen_lines": 0,
        "drop_json_parse": 0,
        "drop_no_eval": 0,
        "drop_opening": 0,
        "drop_mate_or_extreme": 0,
        "drop_sample": 0,
        "drop_dirty_sample": 0,
        "drop_bucket_cap": {sp: 0 for sp in quota_split},
        "drop_reservoir": {sp: 0 for sp in quota_split},
        "reservoir_replaced": {sp: 0 for sp in quota_split},
        "kept_total": 0,
        "kept_by_split": {sp: 0 for sp in quota_split},
        "depth_policy": depth_policy,
        "shuffle_buffer_size": int(shuffle_buffer_size),
        "writer_mode": "two_pass_bucket_reservoir_global_shuffle",
    }

    t0 = time.time()
    completed = False

    try:
        _write_status("running", {"seed": int(seed), "seen_lines": 0})

        def _reserve_index(sp: str, bid):
            st = split_state[sp]
            mode = st["mode"]

            if mode == "bucket":
                b = int(bid)
                cap = int(st["bucket_cap"][b])
                if cap <= 0:
                    stats["drop_bucket_cap"][sp] += 1
                    return None, False, False

                st["seen_bucket"][b] += 1
                seen = int(st["seen_bucket"][b])
                fill = int(st["fill_bucket"][b])

                if fill < cap:
                    st["fill_bucket"][b] = fill + 1
                    idx = int(st["bucket_start"][b] + fill)
                    return idx, False, True

                j = int(rng.integers(0, seen))
                if j < cap:
                    idx = int(st["bucket_start"][b] + j)
                    return idx, True, False

                stats["drop_reservoir"][sp] += 1
                return None, False, False

            # mode == global
            total = int(st["total"])
            st["seen_total"] += 1
            seen = int(st["seen_total"])
            fill = int(st["fill_total"])

            if fill < total:
                st["fill_total"] = fill + 1
                return fill, False, True

            j = int(rng.integers(0, seen))
            if j < total:
                return j, True, False

            stats["drop_reservoir"][sp] += 1
            return None, False, False

        def process_line(line: bytes):
            if not line:
                return

            stats["seen_lines"] += 1

            try:
                obj = orjson.loads(line)
            except Exception:
                stats["drop_json_parse"] += 1
                return

            fen = obj.get("fen")
            evals = obj.get("evals")
            if (not fen) or (not evals):
                return

            best = pick_eval_by_depth(
                evals,
                fixed_depth=fixed_depth,
                min_knodes=min_knodes,
                policy=depth_policy,
            )
            if not best:
                stats["drop_no_eval"] += 1
                return

            pvs = best.get("pvs")
            if not pvs:
                stats["drop_no_eval"] += 1
                return

            pv0 = pvs[0]

            if is_opening_fen(fen):
                stats["drop_opening"] += 1
                return

            if not pv_passes_mate_and_extreme_filters(pv0):
                stats["drop_mate_or_extreme"] += 1
                return

            if "mate" in pv0 and rng.random() > KEEP_MATE_PROB:
                stats["drop_sample"] += 1
                return

            if rng.random() > SAMPLE_PROB:
                stats["drop_sample"] += 1
                return

            sp = split_by_fen(fen, train=SPLIT_RATIO["train"], val=SPLIT_RATIO["val"])
            if sp not in quota_split:
                return

            stm = fen.split()[1]
            y = pv_to_label_value(pv0, stm=stm)

            # Filter dirty samples: repetition/fortress/stalemate noise
            if is_dirty_sample(y, fen):
                stats["drop_dirty_sample"] = stats.get("drop_dirty_sample", 0) + 1
                return

            bid = None
            if split_state[sp]["mode"] == "bucket":
                bid = bucket_id(y, bucket_edges)

            pool_idx, replaced, inserted = _reserve_index(sp, bid)
            if pool_idx is None:
                return

            # encode only if sample selected by reservoir
            b = Board(ensure_full_fen(fen))
            X = to_channels_first(encode_v2(b)).astype(np.float16)

            st = split_state[sp]
            st["X_pool"][pool_idx] = X
            st["y_pool"][pool_idx] = np.float32(y)

            if inserted:
                stats["kept_total"] += 1
                stats["kept_by_split"][sp] += 1
            elif replaced:
                stats["reservoir_replaced"][sp] += 1

            if stats["seen_lines"] % 1000000 == 0:
                dt = time.time() - t0
                print(
                    f"[lines={stats['seen_lines']}] kept={stats['kept_total']} "
                    f"drop_reservoir={stats['drop_reservoir']} time={dt:.1f}s"
                )
                _write_status(
                    "running",
                    {
                        "seen_lines": int(stats["seen_lines"]),
                        "kept_total": int(stats["kept_total"]),
                    },
                )

        with open(zst_path, "rb") as fh:
            dctx = zstd.ZstdDecompressor()
            with dctx.stream_reader(fh) as reader:
                buf = b""
                while True:
                    chunk = reader.read(1 << 20)
                    if not chunk:
                        break
                    buf += chunk

                    while True:
                        nl = buf.find(bytes([10]))
                        if nl < 0:
                            break
                        line = buf[:nl]
                        buf = buf[nl + 1:]
                        process_line(line)

                if buf.strip():
                    process_line(buf.strip())

        # finalize retained stats and strict quota checks
        kept_bucket_by_split = {}
        remain_bucket_by_split = {}

        for sp in quota_split:
            st = split_state[sp]
            if st["mode"] == "bucket":
                fill_bucket = np.asarray(st["fill_bucket"], dtype=np.int64)
                cap_bucket = np.asarray(st["bucket_cap"], dtype=np.int64)
                kept = int(fill_bucket.sum())
                if kept != int(quota_split[sp]):
                    raise RuntimeError(
                        f"Split {sp} underfilled after full pass: kept={kept}, quota={int(quota_split[sp])}. "
                        "Can not build exact dataset with current filters/quota."
                    )
                kept_bucket_by_split[sp] = fill_bucket.tolist()
                remain_bucket_by_split[sp] = (cap_bucket - fill_bucket).tolist()
                stats["kept_by_split"][sp] = kept
            else:
                kept = int(st["fill_total"])
                if kept != int(quota_split[sp]):
                    raise RuntimeError(
                        f"Split {sp} underfilled after full pass: kept={kept}, quota={int(quota_split[sp])}."
                    )
                kept_bucket_by_split[sp] = None
                remain_bucket_by_split[sp] = None
                stats["kept_by_split"][sp] = kept

        stats["kept_total"] = int(sum(stats["kept_by_split"].values()))

        # pass 2: global shuffle then write final shards
        shards_by_split = {}
        for sp in quota_split:
            st = split_state[sp]
            total = int(stats["kept_by_split"][sp])
            perm = rng.permutation(total)
            caps = shard_caps_from_total(total, shard_size)
            shards_by_split[sp] = int(caps.shape[0])

            split_stage = os.path.join(stage_dir, sp)
            hist_stage = os.path.join(split_stage, "hists")
            os.makedirs(split_stage, exist_ok=True)
            os.makedirs(hist_stage, exist_ok=True)

            offset = 0
            for si, cap in enumerate(caps):
                cap = int(cap)
                idx = perm[offset:offset + cap]
                offset += cap

                X_out = np.asarray(st["X_pool"][idx], dtype=np.float16)
                y_out = np.asarray(st["y_pool"][idx], dtype=np.float32)

                X_path = os.path.join(split_stage, f"X_{si:05d}.npy")
                y_path = os.path.join(split_stage, f"y_{si:05d}.npy")
                np.save(X_path, X_out)
                np.save(y_path, y_out)

                png_path = os.path.join(hist_stage, f"hist_{si:05d}.png")
                save_hist_png(y_out, bucket_edges, png_path, f"{sp} shard {si:05d} (n={cap})")

            if offset != total:
                raise RuntimeError(f"Split {sp}: shuffle-write offset {offset} != total {total}")

        # move stage to final output
        for sp in quota_split:
            src = os.path.join(stage_dir, sp)
            dst = os.path.join(out_dir, sp)
            if os.path.exists(dst):
                shutil.rmtree(dst)
            os.replace(src, dst)

        stats["shards_by_split"] = shards_by_split
        stats["kept_bucket_by_split"] = kept_bucket_by_split
        stats["quota_bucket_by_split"] = {
            sp: (None if split_bucket_quota.get(sp) is None else np.asarray(split_bucket_quota[sp], dtype=np.int64).tolist())
            for sp in quota_split
        }
        stats["remain_bucket_by_split"] = remain_bucket_by_split

        completed = True
        _write_status(
            "completed",
            {
                "seen_lines": int(stats["seen_lines"]),
                "kept_total": int(stats["kept_total"]),
                "kept_by_split": stats["kept_by_split"],
            },
        )

        return stats

    except Exception as e:
        _write_status(
            "failed",
            {
                "seen_lines": int(stats.get("seen_lines", 0)),
                "error": str(e),
            },
        )
        raise

    finally:
        _close_pool_maps()
        # keep pool only when failed for debugging; remove on success
        if completed:
            try:
                import shutil
                if os.path.exists(pool_dir):
                    shutil.rmtree(pool_dir)
                if os.path.exists(stage_dir):
                    shutil.rmtree(stage_dir)
            except Exception:
                pass



In [ ]:
result = preprocess_mixed(
    zst_path=RAW_DIR,
    out_dir=PROC_DIR,
    shard_size=SHARD_SIZE,
    quota_split=quota_split,
    bucket_edges=BUCKET_EDGES,
    split_bucket_quota=split_bucket_quota,
    fixed_depth=FIXED_DEPTH,
    min_knodes=MIN_KNODES,
    depth_policy=DEPTH_POLICY,
    shuffle_buffer_size=SHUFFLE_BUFFER_SIZE,
    seed=123
)

print("PROC_DIR:", PROC_DIR)
print(result)



In [ ]:
print('kept by split:', result["kept_by_split"])
print('drop_bucket_cap:', result["drop_bucket_cap"])
print('depth_policy:', result.get("depth_policy"))
print('shuffle_buffer_size:', result.get("shuffle_buffer_size"))
for sp in ["train", "val", "test"]:
    print(f"\n[{sp}] kept/quota/remain")
    print(' kept:', result["kept_bucket_by_split"][sp])
    print(' quota:', result["quota_bucket_by_split"][sp])
    print(' remain:', result["remain_bucket_by_split"][sp])
    if result["quota_bucket_by_split"][sp] is not None:
        k = np.array(result["kept_bucket_by_split"][sp], dtype=np.float64)
        q = np.array(result["quota_bucket_by_split"][sp], dtype=np.float64)
        ratio = np.divide(k, q, out=np.zeros_like(k), where=q>0)
        print(' fill_ratio:', np.round(ratio, 4).tolist())



In [ ]:
import glob, os, numpy as np

def quick_check(split="train", k=3):
    X_files = sorted(glob.glob(os.path.join(PROC_DIR, split, "X_*.npy")))
    y_files = sorted(glob.glob(os.path.join(PROC_DIR, split, "y_*.npy")))
    print(split, "num_shards:", len(X_files), len(y_files))

    if len(X_files) == 0:
        return

    picked = np.unique(np.linspace(0, len(X_files)-1, min(k, len(X_files)), dtype=int))
    for idx in picked:
        X = np.load(X_files[idx], mmap_mode="r")
        y_raw = np.load(y_files[idx], mmap_mode="r")
        y = y_raw.astype(np.float32)

        # Plane 12 là cờ side-to-move (white=1, black=0).
        # Với encode STM-relative, plane này có thể dư thừa thông tin nhưng vẫn hợp lệ.
        stm_white_ratio = float(X[:, 12].mean())

        hist, _ = np.histogram(y, bins=BUCKET_EDGES)
        hist = hist.astype(np.float64)
        hist = hist / max(1.0, hist.sum())
        peak_bin = int(hist.argmax())

        print(
            f" shard {idx:05d}: X{tuple(X.shape)} y{tuple(y.shape)} "
            f"Xdtype={X.dtype} ydtype={y_raw.dtype} "
            f"Xmin/max={int(X.min())}/{int(X.max())} "
            f"ymean={float(y.mean()):.4f} ystd={float(y.std()):.4f} "
            f"ymin/max={float(y.min()):.3f}/{float(y.max()):.3f} "
            f"stm_white_ratio={stm_white_ratio:.3f} peak_bin={peak_bin}"
        )

        abs_y = np.abs(y)
        print(
            "   abs(y) mass:"
            f" <=0.1={float((abs_y <= 0.1).mean()):.3f},"
            f" <=0.2={float((abs_y <= 0.2).mean()):.3f},"
            f" <=0.4={float((abs_y <= 0.4).mean()):.3f},"
            f" >0.7={float((abs_y > 0.7).mean()):.3f}"
        )

    # Drift check giữa shard đầu và shard cuối (simple)
    if len(X_files) >= 2:
        y0 = np.load(y_files[0], mmap_mode="r").astype(np.float32)
        y1 = np.load(y_files[-1], mmap_mode="r").astype(np.float32)
        h0, _ = np.histogram(y0, bins=BUCKET_EDGES)
        h1, _ = np.histogram(y1, bins=BUCKET_EDGES)
        h0 = h0 / max(1.0, h0.sum())
        h1 = h1 / max(1.0, h1.sum())
        l1 = float(np.abs(h0 - h1).sum())
        print(f" {split} shard_drift_l1(first,last)={l1:.4f}")

quick_check("train")
quick_check("val")
quick_check("test")


